In [ ]:
        ##Attribution###
## Implements a multi‑year LST processing workflow for Dhaka (2014–2024).

## Includes cloud masking, radiometric calibration, spectral indices, emissivity estimation, and single‑channel LST retrieval.

## Adapts the semi‑automatic Landsat time‑series workflow developed by Mohiuddin & Mund (2024) for Phnom Penh.

## Builds on established scientific methods for:

## Radiometric calibration — Nill et al. (2019)

## Single‑channel LST retrieval — Jiménez‑Muñoz et al. (2008)

## Emissivity estimation — Sobrino et al. (2008)

## Fractional vegetation cover (FVC) — Carlson & Ripley (1997)

## Extends the workflow with Dhaka‑specific preprocessing and composite generation.

## All Dhaka‑specific modifications, including additional indices, seasonal
## compositing, and export routines, were implemented by Mahbubul Alam.

In [ ]:
# installing libraries
!pip install geemap --upgrade
!pip install geopandas
!pip install osmnx


In [ ]:
# loading necessary libraries
import os
import sys
import ee
import geemap
import osmnx as ox
import geopandas as gpd
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

In [ ]:
ee.Authenticate()
ee.Initialize(project='ee-sanim')


In [ ]:

#study period
year_start = 2014
year_end = 2024
month_start = 1
month_end = 12

# temperature threshold
t_threshold = 20

# cloud filter
max_cloud_cover = 60 # in percentage

aoi = 'projects/ee-sanim/assets/Coredhaka'
roi = ee.FeatureCollection(aoi)

# Min and max NDVI values
ndvi_v = 0.86
ndvi_s = -0.64

# emissivity value (Li et al., 2013)
epsilon_v = 0.985
epsilon_s = 0.97
epsilon_w = 0.99

# coefficients for atmospheric functions (Jiménez‐Muñoz et al. (2008) & Jiménez‐Muñoz et al. (2014))
cs_l8 = [0.04019, 0.02916, 1.01523,
         -0.38333, -1.50294, 0.20324,
         0.00918, 1.36072, -0.27514]
cs_l9 = [0.06518, 0.00683, 1.02717,
         -0.53003, -1.25866, 0.10490,
         -0.01965, 1.36947, -0.24310]


Map = geemap.Map()
Map.centerObject(roi, 10)
Map.addLayer(roi, {}, "Dhaka AOI")
Map

Map(center=[23.789102582046304, 90.4166173461112], controls=(WidgetControl(options=['position', 'transparent_b…

In [ ]:
## Rename bands and clipping
# renaming bands
def fun_bands_l57(img):
       bands = ['B1', 'B2', 'B3', 'B4', 'B5', 'B7']
       thermal_band = ['B6']
       new_bands = ['B', 'G', 'R', 'NIR', 'SWIR1', 'SWIR2']
       new_thermal_bands = ['TIR']
       vnirswir = img.select(bands).multiply(0.0001).rename(new_bands)
       tir = img.select(thermal_band).multiply(0.1).rename(new_thermal_bands)
       return vnirswir.addBands(tir).copyProperties(img, ['system:time_start'])


def fun_bands_l8(img):
       bands = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7']
       thermal_band = ['B10']
       new_bands = ['B', 'G', 'R', 'NIR', 'SWIR1', 'SWIR2']
       new_thermal_bands = ['TIR']
       vnirswir = img.select(bands).multiply(0.0001).rename(new_bands)
       tir = img.select(thermal_band).multiply(0.1).rename(new_thermal_bands)
       return vnirswir.addBands(tir).copyProperties(img, ['system:time_start'])

#clipping
def fun_clip(img):
  clip_img = img.clip(roi)
  return clip_img


In [ ]:
## Masking
# Function to cloud mask Landsat TM, ETM+, OLI_TIRS Surface Reflectance Products (Foga et al., 2017)
def fun_mask_ls_sr(img):
       cloudShadowBitMask = ee.Number(2).pow(3).int()
       cloudsBitMask = ee.Number(2).pow(5).int()
       snowBitMask = ee.Number(2).pow(4).int()
       qa = img.select('pixel_qa')
       mask = qa.bitwiseAnd(cloudShadowBitMask).eq(0).And(
              qa.bitwiseAnd(cloudsBitMask).eq(0)).And(
              qa.bitwiseAnd(snowBitMask).eq(0))
       return img.updateMask(mask)

# Function to mask LST below certain temperature threshold
def fun_mask_T(img):
    mask = img.select('LST').gt(t_threshold)
    return img.updateMask(mask)

In [ ]:
## Matching and calibration
# Radiometric Calibration
def fun_radcal(img):
    radiance = ee.Algorithms.Landsat.calibratedRadiance(img).rename('RADIANCE')
    return img.addBands(radiance)

# L to ee.Image
def fun_l_addband(img):
    l = ee.Image(img.get('L')).select('RADIANCE').rename('L')
    return img.addBands(l)

# Create maxDifference-filter to match TOA and SR products
maxDiffFilter = ee.Filter.maxDifference(
    difference=2 * 24 * 60 * 60 * 1000,
    leftField= 'system:time_start',
    rightField= 'system:time_start'
)

# Define join: Water vapor
join_wv = ee.Join.saveBest(
    matchKey = 'WV',
    measureKey = 'timeDiff'
)

# Define join: Radiance
join_l = ee.Join.saveBest(
    matchKey = 'L',
    measureKey = 'timeDiff'
)

In [ ]:
## Spectral Indices
# NDVI (Rouse et al., 1973)
def fun_ndvi(img):
    ndvi = img.normalizedDifference(['NIR', 'R']).rename('NDVI')
    return img.addBands(ndvi)

#MNDWI (Xu, 2006)
def fun_mndwi(img):
    mndwi = img.normalizedDifference(['G', 'SWIR1']).rename('MNDWI')
    return img.addBands(mndwi)

# IBI (Xu, 2008)
def fun_ibi(img):
    const = ee.Number(2)

    ibi = img.expression(
        '(((const * swir1) / (swir1 + nir))-((nir / (nir + red)) + (green / (green + swir1))))   / (((const * swir1) / (swir1 + nir))+((nir / (nir + red)) + (green / (green + swir1))))',
        {
            'swir1': img.select('SWIR1'),
            'nir': img.select('NIR'),
            'red': img.select('R'),
            'green': img.select('G'),
            'const': const
        }).rename('IBI')
    return img.addBands(ibi)

# Parameter calculation
# Fraction Vegetation Cover (FVC) (Carlson & Ripley, 1997)
def fun_fvc(img):
    fvc = img.expression(
        '((NDVI-NDVI_s)/(NDVI_v-NDVI_s))**2',
        {
            'NDVI': img.select('NDVI'),
            'NDVI_s': ndvi_s,
            'NDVI_v': ndvi_v
        }
    ).rename('FVC')
    return img.addBands(fvc)

# emissivity (Sobrino et al., 2008)
# scale Emissivity (Epsilon) between NDVI_s and NDVI_v
def fun_epsilon_scale(img):
    epsilon_scale = img.expression(
        'epsilon_s+(epsilon_v-epsilon_s)*FVC',
        {
            'FVC': img.select('FVC'),
            'epsilon_s': epsilon_s,
            'epsilon_v': epsilon_v
        }
    ).rename('EPSILON_SCALE')
    return img.addBands(epsilon_scale)

In [4]:
!jupyter nbconvert --ClearMetadataPreprocessor.enabled=True --to notebook --inplace your_notebook.ipynb

[NbConvertApp] WARNING | pattern 'your_notebook.ipynb' matched no files
This application is used to convert notebook files (*.ipynb)
        to various other formats.


Options
The options below are convenience aliases to configurable class-options,
as listed in the "Equivalent to" description-line of the aliases.
To see all configurable class-options for some <cmd>, use:
    <cmd> --help-all

--debug
    set log level to logging.DEBUG (maximize logging output)
    Equivalent to: [--Application.log_level=10]
--show-config
    Show the application's configuration (human-readable format)
    Equivalent to: [--Application.show_config=True]
--show-config-json
    Show the application's configuration (json format)
    Equivalent to: [--Application.show_config_json=True]
--generate-config
    generate default config file
    Equivalent to: [--JupyterApp.generate_config=True]
-y
    Answer yes to any questions instead of prompting.
    Equivalent to: [--JupyterApp.answer_yes=True]
--execute
 